In [31]:
import pandas as pd
from pathlib import Path
import shutil

In [32]:
BASE_DIR      = Path(r"C:\Users\hecto\OneDrive\Escritorio\Personal\iroFactory\35.LinNeo")
DATA_DIR      = BASE_DIR / "biodiversity_data"
NODES_DIR     = DATA_DIR / "taxonomy_nodes_by_rank"
OUTPUT_DIR    = DATA_DIR / "relationships"
CYPHER_OUT    = BASE_DIR / "cypher_import_relationships.cypher"

TAXONOMY_CSV  = DATA_DIR / "gbif_taxonomy.csv"
GEO_CSV       = DATA_DIR / "geographic_hierarchy.csv"
DIST_CSV      = DATA_DIR / "species_geographic_relationships.csv"
SPECIES_CSV   = NODES_DIR / "species_nodes.csv"

NEO4J_IMPORT  = Path(
    r"C:\Users\hecto\.Neo4jDesktop2\Data\dbmss"
    r"\dbms-d1141410-4b1c-481c-b30f-20f8f80f097b\import"
)

In [33]:
# Pares de rangos para relaciones taxonomicas
# Cada par usa las columnas _key del gbif_taxonomy.csv
RANK_PAIRS = [
    {"parent": "Kingdom", "child": "Phylum",  "parent_key": "kingdom_key", "child_key": "phylum_key"},
    {"parent": "Phylum",  "child": "Class",   "parent_key": "phylum_key",  "child_key": "class_key"},
    {"parent": "Class",   "child": "Order",   "parent_key": "class_key",   "child_key": "order_key"},
    {"parent": "Order",   "child": "Family",  "parent_key": "order_key",   "child_key": "family_key"},
    {"parent": "Family",  "child": "Genus",   "parent_key": "family_key",  "child_key": "genus_key"},
    {"parent": "Genus",   "child": "Species", "parent_key": "genus_key",   "child_key": "species_key"},
]

## 1. Geo nodes

In [34]:
def generate_geo_nodes(geo_csv: Path, output_dir: Path) -> tuple[Path, Path]:
    """
    Lee geographic_hierarchy.csv y genera:
        continent_nodes.csv  ->  key (nombre como key), name
        country_nodes.csv    ->  key (country_code), name
    """
    print("Generando nodos geograficos ...")
    df = pd.read_csv(geo_csv, dtype=str)

    # Continentes: el nombre es el identificador unico
    continents = (
        df[["continent"]]
        .rename(columns={"continent": "name"})
        .dropna()
        .drop_duplicates(subset=["name"])
        .sort_values("name")
        .reset_index(drop=True)
    )
    cont_file = output_dir / "continent_nodes.csv"
    continents.to_csv(cont_file, index=False)
    print(f"  Continentes: {len(continents):,}  ->  {cont_file.name}")

    # Paises: country_code es el key
    countries = (
        df[["country_code", "country_name"]]
        .rename(columns={"country_code": "key", "country_name": "name"})
        .dropna(subset=["key"])
        .drop_duplicates(subset=["key"])
        .sort_values("key")
        .reset_index(drop=True)
    )
    # Si country_name esta vacio usamos el propio code como nombre
    countries["name"] = countries["name"].fillna(countries["key"])

    country_file = output_dir / "country_nodes.csv"
    countries.to_csv(country_file, index=False)
    print(f"  Paises:      {len(countries):,}  ->  {country_file.name}")
    print()

    return cont_file, country_file


## 2. Country as part of Continent

In [35]:
def generate_country_continent_rels(geo_csv: Path, output_dir: Path) -> Path:
    """
    Genera country_continent_rels.csv:
        country_key  ->  country_code
        continent    ->  nombre del continente
    """
    print("Generando relaciones Country -> Continent ...")
    df = pd.read_csv(geo_csv, dtype=str)

    rels = (
        df[["country_code", "continent"]]
        .rename(columns={"country_code": "country_key", "continent": "continent_name"})
        .dropna()
        .drop_duplicates()
        .sort_values("country_key")
        .reset_index(drop=True)
    )

    out_file = output_dir / "country_continent_rels.csv"
    rels.to_csv(out_file, index=False)
    print(f"  {len(rels):,} relaciones  ->  {out_file.name}")
    print()
    return out_file

## 3. Full taxon organization

In [36]:
def generate_taxonomic_rels(taxonomy_csv: Path, rank_pairs: list, output_dir: Path) -> dict:
    """
    Para cada par de rangos, extrae las relaciones unicas parent_key -> child_key
    directamente desde gbif_taxonomy.csv.

    Filtra:
        - Filas donde alguno de los dos keys sea 0 o nulo
        - Duplicados

    Retorna dict: "Parent_Child" -> Path del CSV generado.
    """
    print("Cargando gbif_taxonomy.csv para relaciones taxonomicas ...")
    df = pd.read_csv(taxonomy_csv, dtype=str, low_memory=False)
    print(f"  {len(df):,} filas cargadas")
    print()

    rel_files = {}
    print("Generando relaciones taxonomicas:")
    print(f"  {'Relacion':<25} {'Pares unicos':>14}   Archivo")
    print(f"  {'---'*8} {'---'*5}   {'---'*10}")

    for pair in rank_pairs:
        parent     = pair["parent"]
        child      = pair["child"]
        parent_key = pair["parent_key"]
        child_key  = pair["child_key"]
        label      = f"{parent} -> {child}"
        filename   = f"{parent.lower()}_{child.lower()}_rels.csv"

        rels = (
            df[[parent_key, child_key]]
            .rename(columns={parent_key: "parent_key", child_key: "child_key"})
            .dropna()
            .query("parent_key != '0' and parent_key != ''")
            .query("child_key  != '0' and child_key  != ''")
            .drop_duplicates()
            .assign(parent_key=lambda x: x["parent_key"].astype(int))
            .assign(child_key=lambda x: x["child_key"].astype(int))
            .sort_values(["parent_key", "child_key"])
            .reset_index(drop=True)
        )

        out_file = output_dir / filename
        rels.to_csv(out_file, index=False)
        rel_files[f"{parent}_{child}"] = out_file
        print(f"  {label:<25} {len(rels):>14,}   {filename}")

    print()
    return rel_files

## 4. Where to find species

In [37]:
def generate_species_country_rels(dist_csv: Path, species_csv: Path, output_dir: Path) -> Path:
    """
    Genera species_country_rels.csv con todas las especies (sin filtro de reino).

    Cruza species_geographic_relationships.csv con species_nodes.csv para
    asegurarse de que solo se incluyen species_key que existen como nodos.

    Columnas resultantes:
        species_key  ->  key de la especie
        country_key  ->  country_code del pais
    """
    print("Generando relaciones Species -> Country ...")

    dist    = pd.read_csv(dist_csv, dtype=str, low_memory=False)
    species = pd.read_csv(species_csv, dtype=str)

    print(f"  Distribuciones totales en fuente:  {len(dist):,}")

    valid_keys = set(species["key"].astype(str))

    rels = (
        dist[["species_key", "country_code"]]
        .rename(columns={"country_code": "country_key"})
        .dropna()
        .query("species_key != '' and country_key != ''")
        # Solo species_key que existen en species_nodes.csv
        .loc[lambda x: x["species_key"].isin(valid_keys)]
        .drop_duplicates()
        .assign(species_key=lambda x: x["species_key"].astype(int))
        .sort_values(["species_key", "country_key"])
        .reset_index(drop=True)
    )

    out_file = output_dir / "species_country_rels.csv"
    rels.to_csv(out_file, index=False)

    dropped = len(dist) - len(rels)
    print(f"  Relaciones generadas:              {len(rels):,}")
    if dropped > 0:
        print(f"  Descartadas (species_key sin nodo): {dropped:,}")
    print()

    return out_file

## 5. CQL generation

In [38]:
def generate_cypher_relationships(rank_pairs: list, batch_size: int = 10_000) -> str:
    """
    Genera el codigo Cypher completo para:
        1. Constraints de Country y Continent
        2. Importar nodos Country y Continent
        3. Importar todas las relaciones

    Parametros
    ----------
    rank_pairs : list[dict]
        Lista de pares de rangos taxonomicos.
    batch_size : int
        Filas por transaccion en CALL { } IN TRANSACTIONS.

    Notas
    -----
    - Los nodos taxonomicos (Kingdom a Species) deben existir previamente
      (haber corrido cypher_import_taxonomy.cypher).
    - Usa MATCH para los nodos en las relaciones: si un nodo no existe
      la relacion se salta sin error.
    - MERGE en la relacion para ser idempotente.
    """
    sep = "// " + "=" * 60

    lines = [
        sep,
        "// LinNeo -- Import geografia y relaciones",
        sep,
        "//",
        "// REQUISITO: Ejecutar cypher_import_taxonomy.cypher primero.",
        "// Los nodos taxonomicos deben existir antes de crear las relaciones.",
        "//",
        "// INSTRUCCIONES:",
        "// 1. Asegurate de estar usando la base de datos LinNeo",
        "//       :use LinNeo",
        "// 2. Ejecuta cada bloque por separado en el orden indicado",
        "//",
        "",
    ]

    # Constraints geograficos
    lines += [
        "// -- CONSTRAINTS geograficos (ejecutar primero) --",
        "",
        "CREATE CONSTRAINT continent_name IF NOT EXISTS FOR (n:Continent) REQUIRE n.name IS UNIQUE;",
        "CREATE CONSTRAINT country_key IF NOT EXISTS FOR (n:Country) REQUIRE n.key IS UNIQUE;",
        "",
        "",
    ]

    # Nodos geograficos
    lines += [
        "// -- LOAD CSV Continent --",
        "",
        f"LOAD CSV WITH HEADERS FROM 'file:///continent_nodes.csv' AS row",
        "CALL (row) {",
        "  MERGE (n:Continent {name: row.name})",
        f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
        "",
        "// -- LOAD CSV Country --",
        "",
        f"LOAD CSV WITH HEADERS FROM 'file:///country_nodes.csv' AS row",
        "CALL (row) {",
        "  MERGE (n:Country {key: row.key})",
        "  SET n.name = row.name",
        f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
        "",
        "",
    ]

    # Relacion Country -> Continent
    lines += [
        "// -- Country -[PART_OF]-> Continent --",
        "",
        "LOAD CSV WITH HEADERS FROM 'file:///country_continent_rels.csv' AS row",
        "CALL (row) {",
        "  MATCH (country:Country {key: row.country_key})",
        "  MATCH (cont:Continent {name: row.continent_name})",
        "  MERGE (country)-[:PART_OF]->(cont)",
        f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
        "",
        "",
    ]

    # Relaciones taxonomicas
    lines.append("// -- Relaciones taxonomicas (Kingdom a Species) --")
    lines.append("")

    for pair in rank_pairs:
        parent     = pair["parent"]
        child      = pair["child"]
        parent_key = f"{parent.lower()}_key"
        child_key  = f"{child.lower()}_key"
        filename   = f"{parent.lower()}_{child.lower()}_rels.csv"

        lines += [
            f"// {parent} -[HAS_CHILD]-> {child}",
            f"LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row",
            f"CALL (row) {{",
            f"  MATCH (parent:{parent} {{{parent_key}: toInteger(row.parent_key)}})",
            f"  MATCH (child:{child} {{{child_key}: toInteger(row.child_key)}})",
            f"  MERGE (parent)-[:HAS_CHILD]->(child)",
            f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
            "",
        ]

    lines.append("")

    # Species -> Country
    lines += [
        "// -- Species -[FOUND_IN]-> Country --",
        "// Este bloque puede tardar varios minutos por el volumen de datos.",
        "",
        "LOAD CSV WITH HEADERS FROM 'file:///species_country_rels.csv' AS row",
        "CALL (row) {",
        "  MATCH (s:Species {species_key: toInteger(row.species_key)})",
        "  MATCH (c:Country {key: row.country_key})",
        "  MERGE (s)-[:FOUND_IN]->(c)",
        f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
        "",
        "",
    ]

    # Verificacion
    lines += [
        "// -- VERIFICACION --",
        "",
        "MATCH ()-[r]->()",
        "RETURN type(r) AS relacion, count(*) AS total",
        "ORDER BY total DESC;",
    ]

    return "\n".join(lines)

## 6. CSV generation

In [39]:
def copy_to_neo4j_import(files: list[Path], neo4j_import: Path):
    if not neo4j_import.exists():
        print(f"Carpeta import/ no encontrada: {neo4j_import}")
        print(f"Copia manualmente los CSVs desde: {OUTPUT_DIR}")
        print()
        return

    print("Copiando CSVs a Neo4j import/:")
    for src in files:
        dst = neo4j_import / src.name
        shutil.copy2(src, dst)
        print(f"  {src.name} -> {dst}")
    print()

# MAGIC 2

In [40]:
if __name__ == "__main__":

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Nodos geograficos
    cont_file, country_file = generate_geo_nodes(GEO_CSV, OUTPUT_DIR)

    # 2. Relacion Country -> Continent
    cc_rels_file = generate_country_continent_rels(GEO_CSV, OUTPUT_DIR)

    # 3. Relaciones taxonomicas
    tax_rel_files = generate_taxonomic_rels(TAXONOMY_CSV, RANK_PAIRS, OUTPUT_DIR)

    # 4. Relaciones Species -> Country
    sc_rels_file = generate_species_country_rels(DIST_CSV, SPECIES_CSV, OUTPUT_DIR)

    # 5. Generar Cypher
    cypher = generate_cypher_relationships(RANK_PAIRS)
    CYPHER_OUT.write_text(cypher, encoding="utf-8")
    print(f"Cypher guardado en: {CYPHER_OUT}")
    print()

    # 6. Copiar todo a Neo4j import/
    all_files = [
        cont_file,
        country_file,
        cc_rels_file,
        sc_rels_file,
    ] + list(tax_rel_files.values())

    copy_to_neo4j_import(all_files, NEO4J_IMPORT)

Generando nodos geograficos ...
  Continentes: 7  ->  continent_nodes.csv
  Paises:      93  ->  country_nodes.csv

Generando relaciones Country -> Continent ...
  93 relaciones  ->  country_continent_rels.csv

Cargando gbif_taxonomy.csv para relaciones taxonomicas ...
  4,124,085 filas cargadas

Generando relaciones taxonomicas:
  Relacion                    Pares unicos   Archivo
  ------------------------ ---------------   ------------------------------
  Kingdom -> Phylum                    263   kingdom_phylum_rels.csv
  Phylum -> Class                      738   phylum_class_rels.csv
  Class -> Order                     2,657   class_order_rels.csv
  Order -> Family                   19,137   order_family_rels.csv
  Family -> Genus                  250,362   family_genus_rels.csv
  Genus -> Species               3,692,243   genus_species_rels.csv

Generando relaciones Species -> Country ...
  Distribuciones totales en fuente:  624,096
  Relaciones generadas:              519,120
